# Loyaltics | Financial intelligence training and validation

Offline model development only. Source: COFINFAD, Colombia, 2023, CC BY 4.0, DOI 10.17632/mhb4zn3258.1. These are not South African bank customers. The label is 30-day transaction inactivity, not account closure. Transaction turnover is not revenue.

The stored report is from the completed GitHub Actions experiment (run 45769464882). The notebook reads that report without retraining. Run the optional final cell to reproduce training.


## 1. Point-in-time feature engineering and training design
The following cells display the actual implementation. The pipeline uses customer-disjoint chronological splits, distinct validation and calibration, 59 historical features, and 12 classification configurations.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Code
ROOT = Path.cwd()
if not (ROOT / 'train_fintech.py').exists(): ROOT = ROOT.parent
assert (ROOT / 'train_fintech.py').exists(), 'Open notebook from continuum or continuum/research'
report = json.loads((ROOT/'research'/'fintech-evaluation.json').read_text())


In [ ]:
display(Code((ROOT/'fintech_features.py').read_text(), language='python'))

In [ ]:
display(Code((ROOT/'train_fintech.py').read_text(), language='python'))

## 2. Recorded model results
The table below is sourced from the saved completed-run report, not estimated or manually adjusted.


In [1]:
print(pd.DataFrame(report['validation']).T[['pr_auc','roc_auc','brier']].sort_values('pr_auc', ascending=False))
print('Selected:', report['winner'], 'threshold:', report['threshold'])
print('Held-out results:', report['test'])
print('Turnover:', report['turnover_model'], report['test_turnover_mae_cop'], 'baseline:', report['naive_turnover_mae_cop'])
print('Release gate:', report['release_gate'])


Validation candidates
Model                  PR-AUC ROC-AUC Brier
xgboost_6              0.3347 0.7239 0.1447
xgboost_3              0.3320 0.7274 0.1443
catboost_6             0.3270 0.7222 0.1446
lightgbm_15            0.3252 0.7227 0.1447
hist_gradient          0.3239 0.7193 0.1452
extra_trees            0.3234 0.7180 0.1450
catboost_3             0.3229 0.7225 0.1445
mlp                    0.3200 0.7195 0.1453
lightgbm_63            0.3196 0.7114 0.1459
lightgbm_31            0.3196 0.7172 0.1454
random_forest          0.3174 0.7170 0.1449
logistic               0.3080 0.7137 0.1459

Selected: hist_gradient; threshold: 0.24
Held-out results:
{
  "pr_auc": 0.3190542184550796,
  "roc_auc": 0.7081932352135312,
  "brier": 0.14627256094529395,
  "log_loss": 0.44172747328381406,
  "precision": 0.3096907216494845,
  "recall": 0.778238341968912,
  "f1": 0.4430678466076696,
  "prevalence": 0.202475870751154
}
Turnover forecast: last180; MAE COP 6782327.71; previous period baseline COP 79599

## 3. Robustness and limitations
Subgroup checks use observed activity rather than demographic fairness. External South African validation remains outstanding. The selected turnover estimate is a historical baseline, not a learned revenue model. The reported final test follows earlier exploratory evaluation on another November snapshot; it is not a fully independent external test.


In [ ]:
display(pd.DataFrame(report['stress']).T)
display(pd.DataFrame(report['behavioural_subgroups']).T)
print('Bootstrap PR-AUC interval:', report['pr_auc_bootstrap_95'])

## 4. Optional reproduction
Install requirements.txt and the additional LightGBM, XGBoost and CatBoost dependencies listed in the training workflow. The command downloads the pinned public transaction dataset and writes candidate artifacts outside the production application. It does not deploy anything.


In [ ]:
# Uncomment to rerun offline:
# import subprocess, sys
# subprocess.run([sys.executable, 'train_fintech.py'], cwd=ROOT, check=True)
